# 13 · Cold start, debiased evaluation and re-ranking

Guide 12 evaluated models on the usual protocol: interactions held out, every
item rankable, every hit worth the same, the list ordered by score. This guide
changes each of those three assumptions in turn, on MovieLens-100K with the
19 genres as item side information:

- **cold start**: the `item_cold_start` and `user_cold_start` splits,
  `evaluation.candidates` to rank only the cold items, the random floor the
  evaluator logs, and why ties are broken at random;
- **debiased evaluation**: `evaluation.propensity` and the `IPSRecall`,
  `SNIPSRecall`, `IPSDCG` and `SNIPSDCG` estimators next to their naive
  counterparts, and what `clip` does to their variance;
- **re-ranking**: the top-level `rerank` section with `MMR` and `Calibration`,
  applied inside the evaluator, and the accuracy each one gives up for its
  own objective.

Everything runs through the `Evaluator` API (guide 12) with the configuration
section printed next to it.

**Requirements:** `pip install warprec jupyter`. Runs in one to two minutes on
a laptop CPU, most of it training two small sequential models;
`guide_data.movielens_100k()` downloads and converts MovieLens-100K exactly as
guide 1 shows.

In [1]:
import logging
import sys
import warnings
from pathlib import Path

import lightning as L
import pandas as pd
import torch
from loguru import logger

sys.path.insert(0, "..")
from guide_data import GENRES, movielens_100k

folder = movielens_100k()
RUNS = Path("../runs/13-cold-start-debiasing-reranking")
RUNS.mkdir(parents=True, exist_ok=True)
logger.disable("warprec")
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", module="lightning")
warnings.filterwarnings("ignore", module="torch")
warnings.filterwarnings("ignore", message="enable_nested_tensor")

The genres become the side-information file guide 4 builds: the item ID, then
one 0/1 column per genre. The content model reads it to score items, the
re-rankers read it to tell items apart.

In [2]:
items = pd.read_csv(folder / "items.tsv", sep="\t")
genres = pd.DataFrame({"item_id": items.item_id})
for genre in GENRES:
    genres[genre] = items.genres.str.split("|").apply(lambda gs, g=genre: int(g in gs))
genres.to_csv(RUNS / "item-genres.tsv", sep="\t", index=False)
genres.shape

(1682, 20)

A helper builds every model of a configuration the way the pipelines do, and
another evaluates one and averages each per-user metric (guide 12 explains
why the average is a `nanmean`).

In [3]:
from warprec.common.initialize import initialize_datasets
from warprec.data.reader import ReaderFactory
from warprec.evaluation import Evaluator, build_evaluator
from warprec.utils.callback import WarpRecCallback
from warprec.utils.config.config import load_design_configuration
from warprec.utils.registry import model_registry


def load(path):
    """Read a configuration and build its dataset and its (closed-form) models."""
    config = load_design_configuration(path)
    dataset, _, _ = initialize_datasets(
        ReaderFactory.get_reader(config), WarpRecCallback(), config
    )
    models = {
        name: model_registry.get(
            name, params=params, info=dataset.info(), interactions=dataset.train_set
        )
        for name, params in config.models.items()
    }
    return config, dataset, models


def evaluate(evaluator, model, dataset):
    """Evaluate one model; per-user metrics reported as their nanmean."""
    evaluator.evaluate(model, dataset.get_evaluation_dataloader(), "full", dataset)
    return {
        f"{metric}@{k}": value.nanmean().item() if torch.is_tensor(value) else value
        for k, metrics in evaluator.compute_results().items()
        for metric, value in metrics.items()
    }

## Item cold start

`item_cold_start` holds out **every** interaction of a random 10% of the items
(guide 3 shows the split itself). The configuration also restricts the
ranking to those items with `evaluation.candidates: cold`, and compares a
content model (`VSM`, which scores items from their genres) with three
collaborative ones and a `Random` baseline.

In [4]:
print(Path("configs/item-cold-start.yml").read_text())

# Item cold start: every interaction of 10% of the films is held out, and the
# ranking is restricted to those films. Genres are the side information the
# content model scores cold films from.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
  side:
    local_path: ../runs/13-cold-start-debiasing-reranking/item-genres.tsv
splitter:
  test_splitting:
    strategy: item_cold_start
    ratio: 0.1      # a fraction of the items, not of the rows
    seed: 42
models:
  Random: {}
  Pop: {}
  ItemKNN:
    k: 50
    similarity: cosine
  EASE:
    l2: 200.0
  VSM:
    similarity: cosine
    user_profile: tfidf
    item_profile: binary
evaluation:
  top_k: [10]
  metrics: [nDCG, Recall, HitRate]
  candidates: cold  # all (default), cold or warm



In [5]:
config, cold_items, models = load("configs/item-cold-start.yml")
train = cold_items.train_set.get_sparse()
test = cold_items.eval_set.get_sparse()
is_cold = torch.as_tensor(train.getnnz(axis=0) == 0)
print(f"catalogue: {train.shape[1]} items, {int(is_cold.sum())} without training rows")
print(
    f"test: {test.nnz} rows over {(test.getnnz(axis=0) > 0).sum()} items, "
    f"all of them cold: {bool(is_cold[test.getnnz(axis=0) > 0].all())}"
)

catalogue: 1682 items, 168 without training rows
test: 10292 rows over 168 items, all of them cold: True


The held-out items are still in the catalogue. They have no training row, so
an ordinary split would drop them from the mappings and from the test set;
under a cold-start strategy WarpRec keeps them, as empty columns of the
training matrix, without any configuration. The test set therefore holds
nothing but cold items.

### `candidates`: which items are ranked

`candidates` takes `all` (the default), `cold` (items with no training
interaction) or `warm` (the rest). It is an argument of the `Evaluator`, which
removes the other items from every ranking before masking the training items.
When the pool is restricted, the evaluator logs it with the score a model gets
for ranking the pool blindly: `min(1, k / pool)`, the share of a user's cold
test items a random top-k retrieves.

In [6]:
logger.enable("warprec")
evaluators = {
    candidates: Evaluator(
        ["nDCG", "Recall", "HitRate"], [10], train_set=train, candidates=candidates
    )
    for candidates in ["all", "cold", "warm"]
}
logger.disable("warprec")

08-10-2026 12:49:54 - WarpRec - 📢 Ranking restricted to 168 'cold' items. At k=10 a model that scores them all alike already retrieves about 6% of them, so read every result against that floor rather than against zero.


08-10-2026 12:49:54 - WarpRec - 📢 Ranking restricted to 1514 'warm' items. At k=10 a model that scores them all alike already retrieves about 1% of them, so read every result against that floor rather than against zero.


In [7]:
by_pool = pd.DataFrame(
    {
        (candidates, name): evaluate(evaluator, model, cold_items)
        for candidates, evaluator in evaluators.items()
        for name, model in models.items()
    }
).T
by_pool.round(4)

nDCG@10  Recall@10  HitRate@10
all  Random    0.0098     0.0068      0.0763
     Pop       0.0000     0.0000      0.0000
     ItemKNN   0.0000     0.0000      0.0000
     EASE      0.0000     0.0000      0.0000
     VSM       0.0219     0.0188      0.1321
cold Random    0.0775     0.0598      0.4168
     Pop       0.0419     0.0310      0.3169
     ItemKNN   0.0419     0.0310      0.3169
     EASE      0.0419     0.0310      0.3169
     VSM       0.1540     0.1321      0.6090
warm Random    0.0000     0.0000      0.0000
     Pop       0.0000     0.0000      0.0000
     ItemKNN   0.0000     0.0000      0.0000
     EASE      0.0000     0.0000      0.0000
     VSM       0.0000     0.0000      0.0000

Read the three blocks against the floor, not against zero:

- **`all`**: the collaborative models score 0. Every warm item gets some score
  from them and every cold item none, so no cold item ever reaches their top
  10, and the test set contains only cold items. `VSM` and `Random` are the
  only models that ever put one there.
- **`cold`**: `Random` lands on the 6% floor the log announced (Recall@10
  0.0598). `VSM` more than doubles it (0.1321): the genres carry real signal
  about which new films a user will watch. Pop, ItemKNN and EASE score exactly
  the same, 0.0310, below the floor; the next cells show why.
- **`warm`**: every model scores 0. Under `item_cold_start` the test set holds
  only cold items, so a warm pool contains none of them.

### Why the collaborative models sit at the floor

Pop, ItemKNN and EASE score an item from its training interactions. A cold
item is an all-zero column, so each of them gives every cold item exactly the
same score, for every user:

In [8]:
users = torch.arange(train.shape[0])
with torch.no_grad():
    distinct = {
        name: max(
            len(row.unique()) for row in model.predict(user_indices=users)[:, is_cold]
        )
        for name, model in models.items()
    }
pd.Series(distinct, name="distinct scores over the cold items (max per user)")

Random     168
Pop          1
ItemKNN      1
EASE         1
VSM         68
Name: distinct scores over the cold items (max per user), dtype: int64

Their ranking of the cold pool is decided entirely by how ties are broken.
`torch.topk` breaks them by position, which would hand the model the cold items
with the lowest indices, and in MovieLens-100K the lowest IDs are among its
best-known films. `top_k_breaking_ties` (in `warprec.data.ranking`) is what the
evaluator uses instead: it shuffles the tied rows with a seeded generator before
taking the top k. Without a generator it falls back to `topk`, which shows what
the positional rule would report for EASE:

In [9]:
from warprec.data.ranking import restrict_to_candidates, top_k_breaking_ties

with torch.no_grad():
    ease_scores = models["EASE"].predict(user_indices=users).float()
restrict_to_candidates(ease_scores, is_cold)
truth = torch.from_numpy(test.toarray()).float()
has_test = truth.sum(1) > 0


def recall_at_10(indices):
    hits = truth.gather(1, indices).sum(1)
    return (hits[has_test] / truth.sum(1)[has_test]).mean().item()


_, by_position = top_k_breaking_ties(ease_scores, 10)
_, shuffled = top_k_breaking_ties(ease_scores, 10, torch.Generator().manual_seed(42))
item_ids = {index: raw for raw, index in cold_items.get_mappings()[1].items()}
print("by position: raw item IDs", sorted(item_ids[i] for i in by_position[0].tolist()))
print(
    f"test users per item: {truth[:, by_position[0]].sum(0).mean():.0f} for those ten,"
    f" {truth[:, is_cold].sum(0).mean():.0f} over the cold pool"
)
print(f"by position: Recall@10 {recall_at_10(by_position):.4f}")
print(f"shuffled:    Recall@10 {recall_at_10(shuffled):.4f}")

by position: raw item IDs [12, 51, 58, 61, 69, 98, 99, 106, 121, 130]
test users per item: 199 for those ten, 61 over the cold pool
by position: Recall@10 0.1868
shuffled:    Recall@10 0.0310


Ranked by position, EASE would retrieve ten of the most-watched films of the
cold pool and beat the content model (0.1868 against VSM's 0.1321) without
knowing anything about them. Shuffled, it gets 0.0310, the number in the
table above. That is also Pop's and ItemKNN's: the three models tie every cold
item, and each evaluation draws its tie break from the start of the seeded
stream, so all three get the same ranking of the cold pool.

The shuffle is drawn **once per batch**, not once per user, which is what keeps
it affordable on a large catalogue. A model that
ties everything therefore gives the same ten items to every user in a batch,
and with the default batch of 1024 users all 943 users share one list. Its
score is one draw, not an average over draws. Re-evaluating EASE with twenty
evaluator seeds (`seed` is an `Evaluator` argument) shows how wide that draw is:

In [10]:
draws = pd.Series(
    [
        evaluate(
            Evaluator(["Recall"], [10], train_set=train, candidates="cold", seed=seed),
            models["EASE"],
            cold_items,
        )["Recall@10"]
        for seed in range(20)
    ]
)
draws.describe()[["mean", "std", "min", "max"]].round(4)

mean    0.0653
std     0.0281
min     0.0206
max     0.1263
dtype: float64

The draws average 0.065, next to the 6% floor, but range from 0.021 to
0.126, the luckiest one close to VSM's 0.132. So a collaborative model's
number under `candidates: cold` is not a baseline. `Random`, which draws an independent ranking for each user, is: it
is what the docs recommend running next to any cold-start experiment.

The same evaluator comes from the configuration through `build_evaluator`,
which reads `candidates` from the `evaluation` section:

In [11]:
from_config = build_evaluator(config.evaluation, cold_items)
pd.Series(evaluate(from_config, models["VSM"], cold_items), name="VSM").round(4)

nDCG@10       0.1540
Recall@10     0.1321
HitRate@10    0.6090
Name: VSM, dtype: float64

The numbers are the table's for VSM. VSM has ties too: 19 binary genres give
the 168 cold films only 68 distinct scores for a user, so films with the same
genres are ordered by the tie break; with the same seed, every evaluator draws
it the same way. Another `seed` would move VSM by about as much as a tie break
can, so differences of that size between two cold-start runs are noise.

## User cold start

`user_cold_start` holds out every interaction of 10% of the users. Every item
stays warm, so there is nothing for `candidates` to restrict: the evaluation set
contains the cold users alone.

In [12]:
print(Path("configs/user-cold-start.yml").read_text())

# User cold start: every interaction of 10% of the users is held out. The
# items stay warm, so there is no candidate restriction to make.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: user_cold_start
    ratio: 0.1
    seed: 42
models:
  Random: {}
  Pop: {}
  EASE:
    l2: 200.0
evaluation:
  top_k: [10]
  metrics: [nDCG, Recall, HitRate]



In [13]:
config_u, cold_users, models_u = load("configs/user-cold-start.yml")
train_u = cold_users.train_set.get_sparse()
cold = torch.as_tensor(train_u.getnnz(axis=1) == 0).nonzero().flatten()
evaluated = (cold_users.eval_set.get_sparse().getnnz(axis=1) > 0).sum()
print(f"{len(cold)} users without training rows; {evaluated} users evaluated")

94 users without training rows; 94 users evaluated


A model that reads a user's history has nothing to read. For a sequential model
the evaluator asks `train_session.get_user_history_sequences` for each user's
latest items. A cold user's sequence is all padding (the index `n_items`) and
has length 0; a warm user (index 0 here) is shown for comparison:

In [14]:
sequences, lengths = cold_users.train_session.get_user_history_sequences(
    cold[:3].tolist() + [0], 5
)
print(sequences)
print("lengths:", lengths.tolist())

tensor([[1651, 1651, 1651, 1651, 1651],
        [1651, 1651, 1651, 1651, 1651],
        [1651, 1651, 1651, 1651, 1651],
        [ 170,  255,    4,   73,  101]])
lengths: [0, 0, 0, 5]


Two sequential models, SASRec (self-attention) and GRU4Rec (recurrent), are
trained on the warm users exactly as guide 9 trains its iterative models, for
three epochs each.

In [15]:
from warprec.recommenders.trainer.runtime import lightning_runtime
from warprec.utils.registry import params_registry

shared = {
    "embedding_size": 32,
    "dropout_prob": 0.2,
    "reg_weight": 0.0,
    "weight_decay": 0.0,
    "batch_size": 512,
    "epochs": 3,
    "learning_rate": 0.001,
    "neg_samples": 1,
    "max_seq_len": 50,
}
sequential_params = {
    "SASRec": {
        **shared,
        "n_layers": 1,
        "n_heads": 1,
        "inner_size": 64,
        "attn_dropout_prob": 0.2,
    },
    "GRU4Rec": {**shared, "hidden_size": 32, "num_layers": 1},
}


def train(name, params):
    """Build and train a sequential model the way the pipelines do."""
    model = model_registry.get(
        name,
        params=params,
        info=cold_users.info(),
        seed=42,
        interactions=cold_users.train_set,
        sessions=cold_users.train_session,
    )
    optimization = params_registry.get(name, **params).optimization
    model.set_optimization_parameters(optimization.optimizer, optimization.lr_scheduler)
    L.Trainer(
        max_epochs=model.epochs,
        accelerator="cpu",
        **lightning_runtime(optimization, "cpu"),
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    ).fit(
        model,
        train_dataloaders=model.get_dataloader(
            cold_users.train_set, cold_users.train_session
        ),
    )
    return model.eval()


for name, params in sequential_params.items():
    models_u[name] = train(name, params)

Fed those empty sequences, both return real scores, but the same scores for
every cold user; EASE, which multiplies a user's (empty) training row by its
item weights, scores every item 0:

In [16]:
with torch.no_grad():
    seq, seq_len = cold_users.train_session.get_user_history_sequences(
        cold.tolist(), 50
    )
    for name in sequential_params:
        scores_u = models_u[name].predict(
            user_indices=cold, user_seq=seq, seq_len=seq_len
        )
        print(
            f"{name}: finite scores {bool(torch.isfinite(scores_u).all())},",
            f"same row for every cold user {bool((scores_u == scores_u[0]).all())}",
        )
    ease_cold = models_u["EASE"].predict(user_indices=cold)
print("EASE: every cold user's scores are 0:", bool((ease_cold == 0).all()))

SASRec: finite scores True, same row for every cold user True
GRU4Rec: finite scores True, same row for every cold user True
EASE: every cold user's scores are 0: True


In [17]:
pd.DataFrame(
    {
        name: evaluate(
            build_evaluator(config_u.evaluation, cold_users), model, cold_users
        )
        for name, model in models_u.items()
    }
).T.round(4)

,nDCG@10,Recall@10,HitRate@10
Random,0.0527,0.0056,0.3723
Pop,0.5487,0.0906,1.0000
EASE,0.0500,0.0056,0.3723
SASRec,0.4978,0.0921,0.9681
GRU4Rec,0.5034,0.0695,0.9681


Without a history, personalisation is gone. Each sequential model ranks every
cold user by one fixed list, the one its empty-sequence representation
produces, and both lists are good ones: nDCG@10 0.4978 for SASRec and 0.5034
for GRU4Rec against Pop's 0.5487, and SASRec's Recall@10 (0.0921) is level with
Pop's (0.0906). EASE's all-zero rows are ranked by the tie break alone, so it
scores like `Random`. Pop leads on nDCG because popularity is the only thing a
model can still know about a user it has never seen.

## Debiased evaluation

Back to an ordinary split (each user's latest 20%). The `evaluation.propensity`
block gives every item a probability of having been observed, read from its
training count `n_i`: `p_i = (n_i / max_j n_j) ** power`, floored at `clip`. The
four debiased metrics divide each hit by it.

In [18]:
print(Path("configs/debiased.yml").read_text())

# Each user's latest 20% as the test set, scored with the naive metrics and
# their inverse-propensity counterparts.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
  side:
    local_path: ../runs/13-cold-start-debiasing-reranking/item-genres.tsv
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.2
models:
  Pop: {}
  EASE:
    l2: 200.0
  ItemKNN:
    k: 50
    similarity: cosine
  VSM:
    similarity: cosine
    user_profile: tfidf
    item_profile: binary
evaluation:
  top_k: [10]
  metrics: [Recall, nDCG, IPSRecall, SNIPSRecall, IPSDCG, SNIPSDCG, ARP]
  propensity:
    estimator: popularity  # uniform (default) applies no correction
    power: 0.5             # p_i = (n_i / max n) ** power; 0 turns the correction off
    clip: 0.1              # the smallest propensity an item may carry



In [19]:
config_d, temporal, models_d = load("configs/debiased.yml")
train_d = temporal.train_set.get_sparse()

from warprec.evaluation.propensity import build_propensity

propensity = build_propensity(
    train_d.getnnz(axis=0),
    estimator=config_d.evaluation.propensity.estimator,
    power=config_d.evaluation.propensity.power,
    clip=config_d.evaluation.propensity.clip,
)
counts = train_d.getnnz(axis=0)
print(
    f"{len(propensity)} items with {counts.min()} to {counts.max()} training interactions"
)
print(f"{int((propensity == 0.1).sum())} items sit at the clip of 0.1")
pd.Series(propensity.numpy()).describe()[["min", "25%", "50%", "75%", "max"]].round(3)

1613 items with 1 to 518 training interactions
428 items sit at the clip of 0.1


min    0.100
25%    0.100
50%    0.196
75%    0.351
max    1.000
dtype: float64

`build_propensity` is what `build_evaluator` calls; the tensor goes to the
`Evaluator` as `propensity`. The four models are two popularity-heavy ones (Pop,
EASE), ItemKNN, and VSM, which knows only genres and so recommends far less
popular films: `ARP`, the mean training count of the recommended items, says
by how much.

In [20]:
debiased = build_evaluator(config_d.evaluation, temporal)
per_user, table = {}, {}
for name, model in models_d.items():
    debiased.evaluate(model, temporal.get_evaluation_dataloader(), "full", temporal)
    per_user[name] = debiased.compute_results()[10]
    table[name] = {
        m: v.nanmean().item() if torch.is_tensor(v) else v
        for m, v in per_user[name].items()
    }
table = pd.DataFrame(table).T
table.round(4)

,Recall,nDCG,IPSRecall,SNIPSRecall,IPSDCG,SNIPSDCG,ARP
Pop,0.0658,0.1217,0.0758,0.0349,0.6035,0.0166,385.8584
EASE,0.1254,0.2089,0.1968,0.0803,1.3561,0.0379,245.0712
ItemKNN,0.1199,0.1958,0.1715,0.0744,1.1526,0.0350,275.9980
VSM,0.0166,0.0330,0.0478,0.0166,0.4548,0.0074,55.0757


Debiased and naive numbers are not on the same scale, so compare the ratios
between models:

In [21]:
pd.DataFrame(
    {
        "IPSRecall / Recall": table.IPSRecall / table.Recall,
        "EASE over this model, Recall": table.Recall["EASE"] / table.Recall,
        "EASE over this model, SNIPSRecall": table.SNIPSRecall["EASE"]
        / table.SNIPSRecall,
    }
).round(2)

,IPSRecall / Recall,"EASE over this model, Recall","EASE over this model, SNIPSRecall"
Pop,1.15,1.91,2.30
EASE,1.57,1.00,1.00
ItemKNN,1.43,1.05,1.08
VSM,2.88,7.56,4.83


Pop's hits are on items everybody saw, so weighting them by `1/p` barely moves
it (×1.15). VSM's few hits are on rarely seen films and nearly triple (×2.88).
EASE still leads, but by a factor of 4.8 under SNIPSRecall rather than 7.6
under Recall.

Some properties of the four numbers:

- `IPSRecall` divides by the number of relevant items, as Recall does, so a
  user's value can exceed 1.
- `SNIPSRecall` divides by the summed weights of the user's relevant items,
  so it is bounded by 1. It is lower than Recall here because a user's misses
  are mostly rare items, which carry large weights in that sum.
- `IPSDCG` is **not normalised**. nDCG divides by an ideal DCG that does not
  depend on the propensities; reweighting only the numerator would give a
  number that is neither bounded by 1 nor the estimator the literature defines.
  That is why WarpRec has no "IPS-nDCG". `SNIPSDCG` is the bounded form.

In [22]:
pd.DataFrame(
    {
        name: {
            m: res[m][~res[m].isnan()].max().item()
            for m in ["IPSRecall", "SNIPSRecall", "IPSDCG", "SNIPSDCG"]
        }
        for name, res in per_user.items()
    }
).T.round(3).rename(columns=lambda m: f"max {m} over users")

,max IPSRecall over users,max SNIPSRecall over users,max IPSDCG over users,max SNIPSDCG over users
Pop,0.625,0.380,4.945,0.225
EASE,1.606,0.814,8.637,0.429
ItemKNN,1.320,0.814,6.230,0.432
VSM,2.808,1.000,12.498,0.351


`power: 0` makes every propensity 1, which turns each estimator back into its
naive counterpart exactly, a quick check on any dataset:

In [23]:
naive = Evaluator(
    ["Recall", "IPSRecall"],
    [10],
    train_set=train_d,
    propensity=build_propensity(train_d.getnnz(axis=0), power=0.0),
)
naive.evaluate(models_d["EASE"], temporal.get_evaluation_dataloader(), "full", temporal)
power_zero = naive.compute_results()[10]
torch.equal(power_zero["Recall"], power_zero["IPSRecall"])

True

A debiased metric with no propensity is refused rather than reported
uncorrected. With the default `estimator: uniform`, `build_propensity` returns
`None`, and the metric's constructor raises:

In [24]:
try:
    Evaluator(["IPSRecall"], [10], train_set=train_d, propensity=None)
except ValueError as error:
    print(error)

IPSRecall corrects for exposure bias and therefore needs propensities. Set 'evaluation.propensity.estimator' to 'popularity'.


### What `clip` does

With `power: 0.5` the weights `1/p` run from 1 to `(518/1)**0.5 ≈ 23`, and the
default `clip: 0.1` caps them at 10 for the 428 items with five interactions or
fewer. `power: 1` makes the weights `518/n_i`, up to 518, so the clip has far
more to bound.
The per-user spread of IPSRecall@10 at four clips:

In [25]:
spread = {}
for clip in [0.002, 0.01, 0.05, 0.2]:
    evaluator = Evaluator(
        ["IPSRecall", "SNIPSRecall"],
        [10],
        train_set=train_d,
        propensity=build_propensity(train_d.getnnz(axis=0), power=1.0, clip=clip),
    )
    for name in ["EASE", "VSM"]:
        evaluator.evaluate(
            models_d[name], temporal.get_evaluation_dataloader(), "full", temporal
        )
        ips = evaluator.compute_results()[10]["IPSRecall"]
        ips = ips[~ips.isnan()]
        spread[(clip, name)] = {
            "IPSRecall mean": ips.mean().item(),
            "std over users": ips.std().item(),
            "max over users": ips.max().item(),
        }
pd.DataFrame(spread).T.rename_axis(["clip", "model"]).round(3)

IPSRecall mean  std over users  max over users
clip  model                                                
0.002 EASE            0.336           0.505           5.299
      VSM             0.201           0.846          12.627
0.010 EASE            0.336           0.505           5.299
      VSM             0.185           0.788          12.627
0.050 EASE            0.336           0.505           5.299
      VSM             0.135           0.514           8.236
0.200 EASE            0.309           0.388           2.917
      VSM             0.067           0.242           4.942

EASE changes little until the clip reaches 0.2 (mean 0.336, then 0.309): its
hits are mostly on items popular enough to sit above the clip. VSM's score is
carried by hits on rare films: with almost no clipping one user scores 12.6
and the per-user standard deviation is 0.846; at 0.2 the maximum is 4.9, the
deviation 0.242, and the mean has fallen from 0.201 to 0.067. EASE's lead goes
from 1.7× to 4.6× on the clip alone. A higher clip buys a lower variance with
a bias towards the naive metric (`clip: 1` *is* the naive metric), so fix it
before comparing models, and prefer the SNIPS form, which the docs recommend
for its stability.

## Re-ranking

A re-ranker reorders the head of each ranking against a second objective. It
is a top-level `rerank` section, not part of `evaluation`, because it changes
what is recommended: the pipelines pass it to the evaluator and to the
recommendations they write, so both see the same lists. `pool` is how many of
the model's top items are reconsidered; positions past it keep the model's
order.

In [26]:
print(Path("configs/rerank-mmr.yml").read_text())

# The same data and EASE, with MMR reordering the top 100 of every list.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
  side:
    local_path: ../runs/13-cold-start-debiasing-reranking/item-genres.tsv
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.2
models:
  EASE:
    l2: 200.0
rerank:            # top level: it changes what is recommended, not how it is measured
  name: MMR        # MMR or Calibration
  pool: 100        # how many of the top candidates are reordered
  params:
    diversity: 0.3
evaluation:
  top_k: [10]
  metrics: [nDCG, Recall, ItemCoverage, Gini, ShannonEntropy, SRecall]



In [27]:
print(Path("configs/rerank-calibration.yml").read_text())

# Only the rerank section differs from rerank-mmr.yml.
rerank:
  name: Calibration
  pool: 100
  params:
    weight: 0.5      # 0 keeps the model's order, 1 ignores its scores
    smoothing: 0.01  # pulls the list's genre mix towards the user's before the KL



`build_reranker` turns the section into a re-ranker for a dataset, reading the
item features from its side information (and, for Calibration, each user's
genre mix from the training matrix); `build_evaluator` takes it as a third
argument. This is the call the design and eval pipelines make:

In [28]:
from warprec.recommenders.reranking import build_reranker

config_r, _, _ = load("configs/rerank-mmr.yml")
mmr = build_reranker(config_r.rerank, temporal)
print(type(mmr).__name__, mmr.pool, mmr.diversity)

MMR 100 0.3


Both objectives order items by what they are, so a dataset without side
information is refused rather than silently ranked by score:

In [29]:
try:
    build_reranker(config_r.rerank, cold_users)
except ValueError as error:
    print(error)

The 'MMR' re-ranker orders items by what they are, so the dataset needs side information. Configure 'reader.side'.


`MMR` takes at each step the item with the best mix of relevance and
dissimilarity to what the list already holds; `diversity` is the weight of the
second (0 leaves the ranking untouched, 1 ignores the model). `Calibration`
makes the genre mix of the list match the user's history, measured by a
Kullback–Leibler divergence; `weight` is the weight of the divergence, and
`smoothing` mixes a little of the user's distribution into the list's so the
divergence stays finite while a genre is missing. Here each is applied to EASE
at three weights, with the same rest of the `rerank` section built from
`RerankConfig`:

In [30]:
from warprec.utils.config import RerankConfig

rerankers = {"EASE alone": None}
for diversity in [0.3, 0.6, 0.9]:
    rerankers[f"MMR diversity={diversity}"] = build_reranker(
        RerankConfig(name="MMR", pool=100, params={"diversity": diversity}), temporal
    )
for weight in [0.3, 0.6, 0.9]:
    rerankers[f"Calibration weight={weight}"] = build_reranker(
        RerankConfig(name="Calibration", pool=100, params={"weight": weight}), temporal
    )
ease = models_d["EASE"]
reranked = pd.DataFrame(
    {
        label: evaluate(
            build_evaluator(config_r.evaluation, temporal, rr), ease, temporal
        )
        for label, rr in rerankers.items()
    }
).T
reranked.round(4)

,nDCG@10,Recall@10,ItemCoverage@10,Gini@10,ShannonEntropy@10,SRecall@10
EASE alone,0.2089,0.1254,443.0,0.9076,5.3049,0.3156
MMR diversity=0.3,0.2018,0.1197,445.0,0.9076,5.3043,0.3207
MMR diversity=0.6,0.1766,0.1026,462.0,0.9043,5.3339,0.2862
MMR diversity=0.9,0.1571,0.0820,488.0,0.8958,5.4077,0.2455
Calibration weight=0.3,0.2100,0.1271,441.0,0.9071,5.3100,0.3290
Calibration weight=0.6,0.1908,0.1228,450.0,0.9008,5.3710,0.3315
Calibration weight=0.9,0.1587,0.1051,499.0,0.8862,5.5034,0.3142


Both re-rankers give up accuracy as their weight grows; at 0.3 Calibration costs
none here. The catalogue-level metrics (`ItemCoverage`, `Gini`,
`ShannonEntropy`) move in the right direction, modestly (443 films recommended
to someone without re-ranking, at most 499 with), and `SRecall` drops as MMR's
weight grows, because it counts genres among hits and MMR loses hits. None of
them is what either re-ranker optimises. MMR
makes each list diverse; Calibration makes each list look like its user. Those
are per-list properties, so they are measured on the lists themselves. A
re-ranker is a callable: `reranker(scores, k, user_indices)` returns the top-k
values and items, which is what the evaluator calls after masking the training
items.

In [31]:
from warprec.data.ranking import mask_seen_pairs

with torch.no_grad():
    scores = ease.predict(user_indices=users).float()
mask_seen_pairs(scores, train_d)

features = torch.as_tensor(temporal.train_set.get_side_sparse().toarray()).float()
unit = features / features.norm(dim=1, keepdim=True).clamp(min=1e-12)
mix = features / features.sum(dim=1, keepdim=True).clamp(min=1e-12)
history = torch.as_tensor(train_d @ mix.numpy()).float()
history = history / history.sum(dim=1, keepdim=True).clamp(min=1e-12)


def list_profile(top):
    """Mean intra-list genre dissimilarity, and miscalibration against the history."""
    k = top.shape[1]
    similarity = torch.bmm(unit[top], unit[top].transpose(1, 2))
    ild = 1 - (similarity.sum(dim=(1, 2)) - k) / (k * (k - 1))
    shown = 0.99 * mix[top].mean(dim=1) + 0.01 * history
    kl = (history * (history.clamp(min=1e-12) / shown.clamp(min=1e-12)).log()).sum(
        dim=1
    )
    return {
        "intra-list diversity": ild.mean().item(),
        "miscalibration (KL)": kl.mean().item(),
    }


def top10(rr):
    return (
        torch.topk(scores, 10, dim=1).indices
        if rr is None
        else rr(scores, 10, users)[1]
    )


profiles = pd.DataFrame(
    {label: list_profile(top10(rr)) for label, rr in rerankers.items()}
).T
profiles.join(reranked["nDCG@10"]).round(4)

,intra-list diversity,miscalibration (KL),nDCG@10
EASE alone,0.7271,0.6825,0.2089
MMR diversity=0.3,0.7994,0.5248,0.2018
MMR diversity=0.6,0.8970,0.5441,0.1766
MMR diversity=0.9,0.9179,0.5483,0.1571
Calibration weight=0.3,0.7422,0.5097,0.2100
Calibration weight=0.6,0.7453,0.3379,0.1908
Calibration weight=0.9,0.7440,0.1474,0.1587


Each re-ranker is best at its own objective. MMR raises the intra-list
diversity (one minus the mean genre cosine between two films of a list) from
0.727 to 0.918, while Calibration leaves it almost where it was. Calibration
cuts the miscalibration (the divergence from the user's genre mix, with Steck's
0.01 smoothing) from 0.683 to 0.147; MMR reduces it too, since a more varied
list is closer to most histories, but stays between 0.52 and 0.55 however
high its weight. Neither is free: nDCG@10 drops to 0.157 (MMR) and 0.159 (Calibration)
at weight 0.9.

`smoothing` changes what Calibration aims at: at weight 0.6, a smaller value
makes a missing genre cost more, a larger one forgives it. Measured with the
same fixed 0.01 as above:

In [32]:
pd.DataFrame(
    {
        f"smoothing={smoothing}": list_profile(
            top10(
                build_reranker(
                    RerankConfig(
                        name="Calibration",
                        params={"weight": 0.6, "smoothing": smoothing},
                    ),
                    temporal,
                )
            )
        )
        for smoothing in [0.001, 0.01, 0.1]
    }
).T.round(4)

,intra-list diversity,miscalibration (KL)
smoothing=0.001,0.7494,0.2963
smoothing=0.01,0.7453,0.3379
smoothing=0.1,0.7437,0.4448


Less smoothing gives lists closer to the history (0.296 at 0.001, 0.445 at
0.1) at no cost in diversity.

Two things to keep in mind when configuring one:

- the weight trades accuracy for the second objective with no free lunch past
  small values, so treat it as a hyperparameter; but validation **during** a
  hyperparameter search ranks by score alone (the re-ranker applies to the
  reported evaluation and the written recommendations), so tuning it means
  comparing runs;
- the re-rankers are greedy, one step per position of the cutoff, over a
  `pool` × `pool` comparison, so the pool is a budget rather than a quality knob.

## Summary

| section | key | what it does |
|---|---|---|
| `splitter.test_splitting` | `strategy: item_cold_start` / `user_cold_start`, `ratio`, `seed` | hold out every interaction of a fraction of the items / users; they stay in the catalogue |
| `evaluation` | `candidates` | `all` (default), `cold` or `warm`: which items are ranked; a restricted pool logs its random floor `min(1, k/pool)` |
| `evaluation.propensity` | `estimator` | `uniform` (default, no correction) or `popularity` |
| | `power` | exponent of `n_i / max n` (default 0.5); 0 turns the correction off |
| | `clip` | smallest propensity (default 0.1); bounds the weights, trading variance for bias |
| `evaluation.metrics` | `IPSRecall`, `SNIPSRecall`, `IPSDCG`, `SNIPSDCG` | debiased estimators; require `estimator: popularity` |
| `rerank` | `name` | `MMR` or `Calibration`; needs `reader.side` |
| | `pool` | how many top items are reordered (default 100) |
| | `params` | MMR `diversity` (0.3); Calibration `weight` (0.5), `smoothing` (0.01) |

The docs pages: [Cold-Start Protocols](https://warprec.readthedocs.io/en/latest/evaluation/cold-start/),
[Debiased Evaluation](https://warprec.readthedocs.io/en/latest/evaluation/debiased/)
with the [debiased metrics](https://warprec.readthedocs.io/en/latest/evaluation/metrics/debiased/),
and [Re-Ranking](https://warprec.readthedocs.io/en/latest/recommenders/reranking/).
Guide 14 evaluates a saved checkpoint or an external recommendation file, where
the same `candidates`, `propensity` and `rerank` sections apply.